# Extraction Walkthrough — Direct Ollama (no client wrapper)

Companion to `extraction_walkthrough.ipynb`. Sends the SAME prompts that docling-graph would assemble, but POSTs them DIRECTLY to Ollama (`/api/chat`) — bypassing the docling-graph FastAPI service AND the in-process `OllamaChatClient` it injects into `PipelineConfig(llm_client=...)`.

**Why this notebook exists:** historically LiteLLM's Ollama adapter dropped the response body for `gpt-oss:120b + format=json + think=low/medium/high` — Ollama returned valid JSON, LiteLLM handed docling-graph an empty `content` string. The OllamaPool refactor (Chunks 1-5, 2026-04-28) replaced LiteLLM with our own `OllamaChatClient` that talks `/v1/chat/completions` directly, so that specific failure mode is gone.

This notebook is still useful as a **reference implementation** of "what Ollama returns when nothing is in the way" — handy when debugging a new model, a new schema, or any future client-layer regression. The output here is the absolute floor: anything `OllamaChatClient` produces in production should match (modulo schema-vs-format-json mode, which this notebook also exposes via `FORMAT_MODE`).

**What it does:**
1. Reuses docling-graph's library functions (`DocumentChunker`, `build_delta_node_catalog`, `build_delta_semantic_guide`, `format_batch_markdown`, `get_delta_batch_prompt`) — the prompts are identical to what the service would build.
2. POSTs `{"model": ..., "messages": [...], "format": "json", "think": "low"}` directly to Ollama at `10.0.1.121:11434/api/chat`.
3. Parses the response `content` field as JSON.
4. Validates the parsed JSON against the pass's Pydantic template class (same schema docling-graph would validate against).
5. Aggregates across all 11 entity-emitting passes with the same canonicalize + merge logic as `extraction_walkthrough.ipynb` §16.

**Skipped:** `system_links` (relationships_only pass) — that pass uses a more elaborate prompt assembly with upstream-entity preamble injection that's harder to replicate one-shot. Drop me a line if you want it.

**Prerequisite:** Jupyter container reachable to `10.0.1.121:11434` (already validated). For multi-host fan-out you can swap `OLLAMA_URL` for any URL in your `OLLAMA_LLM_BASE_URLS` JSON array — this notebook hits one URL at a time, so no pool routing here.


## §1 Configuration

In [ ]:
import json
import time
import urllib.request
from pprint import pprint

# Direct Ollama endpoint. To match production fan-out, the docling-graph
# service reads its URL pool from the JSON-array env var
# OLLAMA_LLM_BASE_URLS=["http://10.0.1.121:11434","http://10.0.1.122:11434",...]
# (cascade: plural → singular OLLAMA_LLM_BASE_URL → OLLAMA_BASE_URL).
# This notebook hits one URL at a time — pick any entry in the pool below.
OLLAMA_URL    = "http://10.0.1.121:11434/api/chat"
TARGET_MODEL  = "gemma4:31b"   # try also "gemma4:31b" for an apples-to-apples comparison
TARGET_THINK  = "low"            # "low" | "medium" | "high" for gpt-oss; "false" for non-thinking models

# Format mode: matches docling-graph's DOCLING_GRAPH_FORCE_JSON_MODE behavior.
#   "json"   — loose: Ollama enforces JSON-validity only (matches FORCE_JSON_MODE=true)
#   "schema" — strict: Ollama enforces JSON-Schema-conforming output token-by-token
#              using the pass's Pydantic schema (matches FORCE_JSON_MODE=false)
FORMAT_MODE = "json"  # change to "schema" for strict-grammar mode

# Match docling-graph's chunker + batcher config (config_builder.py).
CHUNK_MAX_TOKENS     = 512
LLM_BATCH_TOKEN_SIZE = 1024
TOKENIZER_NAME       = "sentence-transformers/all-MiniLM-L6-v2"

# Same fixture as extraction_walkthrough.ipynb so results are directly comparable.
FAKE_TEXT = (
    "The Patriot AN/MPQ-65 is a multi-function phased-array fire-control "
    "radar deployed by the U.S. Army with the Patriot air-defense system. "
    "It operates in C-band at a nominal carrier frequency of 5500 MHz, "
    "with a peak transmitter output of 750 kW and a 35 dBi peak antenna "
    "gain. The array provides a 1.5 degree azimuth beamwidth.\n"
    "\n"
    "By contrast, the AN/SPY-6(V)1 air and missile defense radar (AMDR) "
    "operates in S-band around 3300 MHz with a peak transmit power of "
    "approximately 1500 kW. Its active electronically scanned array "
    "delivers 42 dBi gain with a 0.9 degree beamwidth and is integrated "
    "into the Aegis Combat System on Flight III destroyers.\n"
    "\n"
    "The MIM-104F Patriot Advanced Capability-3 (PAC-3) is the missile "
    "interceptor paired with the AN/MPQ-65 fire-control radar and is "
    "operational with the U.S. Army. The PAC-3 has a body length of "
    "5.2 meters and a body diameter of 0.255 meters, with a total launch "
    "mass of 316 kg. It achieves a maximum intercept range of 35 km and "
    "engages targets up to 25 km altitude, with a minimum engagement "
    "altitude of 0.05 km. The missile uses active radar homing guidance "
    "and reaches Mach 5 (approximately 1700 m/s). Its single-stage solid "
    "rocket booster produces 100 kN of thrust over a 2.5-second burn.\n"
    "\n"
    "The RIM-174 Standard Missile 6 (SM-6) Block IA is the air and "
    "missile defense interceptor paired with the AN/SPY-6(V)1 AESA radar "
    "on Aegis-equipped destroyers. The SM-6 has a body length of 6.55 "
    "meters and a diameter of 0.34 meters with a total launch mass of "
    "1500 kg. Maximum intercept range exceeds 240 km. The Mark 72 "
    "booster delivers approximately 290 kN thrust over a 6-second burn, "
    "after which the dual-pulse Mark 104 sustainer provides extended "
    "cruise. The SM-6 employs semi-active radar homing with terminal "
    "active radar guidance and achieves Mach 3.5 (approximately 1190 "
    "m/s)."
)

print(f"OLLAMA_URL   : {OLLAMA_URL}")
print(f"TARGET_MODEL : {TARGET_MODEL}")
print(f"TARGET_THINK : {TARGET_THINK}")
print(f"fixture text : {len(FAKE_TEXT)} chars")


## §2 Build the DoclingDocument

In [2]:
def build_docling_document(text, name="synthetic-fixture"):
    return {
        "schema_name": "DoclingDocument",
        "version": "1.0.0",
        "name": name,
        "origin": {"mimetype": "text/plain", "binary_hash": 1, "filename": "smoke.txt"},
        "furniture": {"name": "_root_", "self_ref": "#/furniture", "children": []},
        "body": {"name": "_root_", "self_ref": "#/body",
                 "children": [{"$ref": "#/texts/0"}]},
        "groups": [], "pictures": [], "tables": [],
        "key_value_items": [], "form_items": [], "pages": {},
        "texts": [{"self_ref": "#/texts/0", "parent": {"$ref": "#/body"},
                   "label": "text", "prov": [], "orig": text, "text": text}],
    }

doc = build_docling_document(FAKE_TEXT)
print(f"DoclingDocument built: {len(doc['texts'])} text(s), origin={doc['origin']['filename']}")


DoclingDocument built: 1 text(s), origin=smoke.txt


## §3 Direct-Ollama caller

Two helper functions:

1. **`build_pass_prompt(pass_name, doc)`** — reproduces docling-graph's prompt assembly using the SAME library functions (`DocumentChunker`, `build_delta_node_catalog`, `build_catalog_prompt_block`, `build_delta_semantic_guide`, `format_batch_markdown`, `get_delta_batch_prompt`). Replaces the library's default system prompt with the source-of-truth `DELTA_SYSTEM_PROMPT` from `ontology_bundles/_shared/prompt_rules.py`, matching docling-graph's main.py behavior.

2. **`call_ollama_direct(pass_name, doc, model, think)`** — POSTs `{"messages": [SYSTEM, USER], "format": "json", "think": ...}` straight to Ollama's `/api/chat`. Returns the parsed JSON content (validated against the pass's Pydantic template class).


In [3]:
from importlib import import_module
from docling_core.types.doc import DoclingDocument
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.catalog import build_delta_node_catalog
from docling_graph.core.extractors.contracts.delta.schema_mapper import (
    build_catalog_prompt_block, build_delta_semantic_guide,
)
from docling_graph.core.extractors.contracts.delta.prompts import (
    get_delta_batch_prompt, format_batch_markdown,
)
from ontology_bundles._shared.prompt_rules import DELTA_SYSTEM_PROMPT

PASS_MODULES = {
    "radar_identity":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_identity",       "RadarIdentityPass",       "radar_systems"),
    "radar_power_rf":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_power_rf",       "RadarPowerRfPass",        "radar_systems"),
    "radar_antenna":        ("ontology_bundles.air_defense_v3.extraction_schemas.radar_antenna",        "RadarAntennaPass",        "radar_systems"),
    "radar_timing":         ("ontology_bundles.air_defense_v3.extraction_schemas.radar_timing",         "RadarTimingPass",         "radar_systems"),
    "radar_modulation":     ("ontology_bundles.air_defense_v3.extraction_schemas.radar_modulation",     "RadarModulationPass",     "radar_systems"),
    "missile_identity":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_identity",     "MissileIdentityPass",     "missile_systems"),
    "missile_kinematics":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_kinematics",   "MissileKinematicsPass",   "missile_systems"),
    "missile_guidance":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_guidance",     "MissileGuidancePass",     "missile_systems"),
    "missile_airframe":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_airframe",     "MissileAirframePass",     "missile_systems"),
    "missile_speed_timing": ("ontology_bundles.air_defense_v3.extraction_schemas.missile_speed_timing", "MissileSpeedTimingPass",  "missile_systems"),
    "missile_propulsion":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_propulsion",   "MissilePropulsionPass",   "missile_systems"),
}


def build_pass_prompt(pass_name, doc, batch_index=0):
    """Assemble the SYSTEM + USER prompt for one pass — identical to what
    docling-graph's main.py would compose internally."""
    if pass_name not in PASS_MODULES:
        raise ValueError(f"Unknown pass_name: {pass_name!r}")

    mod_path, cls_name, _ = PASS_MODULES[pass_name]
    template_cls = getattr(import_module(mod_path), cls_name)

    docling_doc = DoclingDocument.model_validate(doc)
    chunker = DocumentChunker(
        tokenizer_name=TOKENIZER_NAME,
        chunk_max_tokens=CHUNK_MAX_TOKENS,
        merge_peers=True,
    )
    chunks = chunker.chunk_document(docling_doc)
    token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
    batch_plan = chunk_batches_by_token_limit(
        chunks, token_counts, max_batch_tokens=LLM_BATCH_TOKEN_SIZE,
    )
    if batch_index >= len(batch_plan):
        raise IndexError(f"batch_index={batch_index} out of range ({len(batch_plan)} batches)")

    selected = batch_plan[batch_index]
    batch_markdown = format_batch_markdown([t for _, t, _ in selected])

    catalog        = build_delta_node_catalog(template_cls)
    catalog_block  = build_catalog_prompt_block(catalog)
    schema_dict    = template_cls.model_json_schema()
    semantic_guide = build_delta_semantic_guide(template_cls, schema_dict)

    first_chunk = chunks[0].strip() if chunks else ""
    global_context = (first_chunk[:600] + ("..." if len(first_chunk) > 600 else "")) \
        if first_chunk else None

    raw = get_delta_batch_prompt(
        batch_markdown=batch_markdown,
        schema_semantic_guide=semantic_guide,
        path_catalog_block=catalog_block,
        batch_index=batch_index,
        total_batches=len(batch_plan),
        global_context=global_context,
        already_found=None,
    )
    # Override the library's default SYSTEM prompt with the source-of-truth
    # (matches docling-graph/app/main.py).
    return DELTA_SYSTEM_PROMPT, raw["user"], template_cls


def call_ollama_direct(pass_name, doc, model=None, think=None, timeout=600):
    """POST a single chat completion to Ollama for one pass. Returns:

        {
            "elapsed_seconds": float,
            "raw_content":     str,           # the model's content field
            "raw_thinking":    str,           # the model's thinking field (if any)
            "parsed":          dict | None,   # JSON-parsed content
            "validated":       BaseModel | None,  # Pydantic-validated against pass template
            "validation_error": str | None,
            "request":         dict,          # exact body sent
        }
    """
    model = model or TARGET_MODEL
    think = think if think is not None else TARGET_THINK

    system, user, template_cls = build_pass_prompt(pass_name, doc)

    # Format mode: "json" (loose) sends format="json"; "schema" (strict) sends
    # the pass's JSON Schema so Ollama enforces shape token-by-token.
    if FORMAT_MODE == "schema":
        ollama_format = template_cls.model_json_schema()
    elif FORMAT_MODE == "json":
        ollama_format = "json"
    else:
        raise ValueError(f"FORMAT_MODE must be 'json' or 'schema', got {FORMAT_MODE!r}")

    body = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "stream": False,
        "format": ollama_format,
        "options": {"temperature": 0},
    }
    if think:
        body["think"] = think

    payload = json.dumps(body).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.monotonic()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        resp = json.loads(r.read())
    elapsed = time.monotonic() - t0

    msg = resp.get("message", {}) or {}
    raw_content  = msg.get("content", "") or ""
    raw_thinking = msg.get("thinking", "") or ""

    parsed, validated, vrr = None, None, None
    if raw_content.strip():
        try:
            parsed = json.loads(raw_content)
        except json.JSONDecodeError as exc:
            vrr = f"json.loads failed: {exc}"
        if parsed is not None:
            try:
                validated = template_cls.model_validate(parsed)
            except Exception as exc:
                vrr = f"pydantic validate failed: {exc}"
    else:
        vrr = "empty content from Ollama"

    return {
        "elapsed_seconds":  elapsed,
        "raw_content":      raw_content,
        "raw_thinking":     raw_thinking,
        "parsed":           parsed,
        "validated":        validated,
        "validation_error": vrr,
        "request":          body,
    }


# Smoke test — make sure the helper compiles + Ollama is reachable.
print("PASS_MODULES keys:", list(PASS_MODULES))


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PASS_MODULES keys: ['radar_identity', 'radar_power_rf', 'radar_antenna', 'radar_timing', 'radar_modulation', 'missile_identity', 'missile_kinematics', 'missile_guidance', 'missile_airframe', 'missile_speed_timing', 'missile_propulsion']


## §4 `radar_identity`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_rid = call_ollama_direct("radar_identity", doc)
print(f"-- elapsed: {res_rid['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_rid['raw_thinking'])} chars): "
      f"{res_rid['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_rid['raw_content'])} chars): "
      f"{res_rid['raw_content'][:500]!r}")
print()
if res_rid["validation_error"]:
    print(f"⚠ validation: {res_rid['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_rid["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

## §5 `radar_power_rf`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_rpr = call_ollama_direct("radar_power_rf", doc)
print(f"-- elapsed: {res_rpr['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_rpr['raw_thinking'])} chars): "
      f"{res_rpr['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_rpr['raw_content'])} chars): "
      f"{res_rpr['raw_content'][:500]!r}")
print()
if res_rpr["validation_error"]:
    print(f"⚠ validation: {res_rpr['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_rpr["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §6 `radar_antenna`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_ran = call_ollama_direct("radar_antenna", doc)
print(f"-- elapsed: {res_ran['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_ran['raw_thinking'])} chars): "
      f"{res_ran['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_ran['raw_content'])} chars): "
      f"{res_ran['raw_content'][:500]!r}")
print()
if res_ran["validation_error"]:
    print(f"⚠ validation: {res_ran['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_ran["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §7 `radar_timing`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_rt = call_ollama_direct("radar_timing", doc)
print(f"-- elapsed: {res_rt['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_rt['raw_thinking'])} chars): "
      f"{res_rt['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_rt['raw_content'])} chars): "
      f"{res_rt['raw_content'][:500]!r}")
print()
if res_rt["validation_error"]:
    print(f"⚠ validation: {res_rt['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_rt["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §8 `radar_modulation`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_rm = call_ollama_direct("radar_modulation", doc)
print(f"-- elapsed: {res_rm['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_rm['raw_thinking'])} chars): "
      f"{res_rm['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_rm['raw_content'])} chars): "
      f"{res_rm['raw_content'][:500]!r}")
print()
if res_rm["validation_error"]:
    print(f"⚠ validation: {res_rm['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_rm["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §9 `missile_identity`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mid = call_ollama_direct("missile_identity", doc)
print(f"-- elapsed: {res_mid['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mid['raw_thinking'])} chars): "
      f"{res_mid['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mid['raw_content'])} chars): "
      f"{res_mid['raw_content'][:500]!r}")
print()
if res_mid["validation_error"]:
    print(f"⚠ validation: {res_mid['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mid["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §10 `missile_kinematics`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mki = call_ollama_direct("missile_kinematics", doc)
print(f"-- elapsed: {res_mki['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mki['raw_thinking'])} chars): "
      f"{res_mki['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mki['raw_content'])} chars): "
      f"{res_mki['raw_content'][:500]!r}")
print()
if res_mki["validation_error"]:
    print(f"⚠ validation: {res_mki['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mki["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §11 `missile_guidance`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mg = call_ollama_direct("missile_guidance", doc)
print(f"-- elapsed: {res_mg['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mg['raw_thinking'])} chars): "
      f"{res_mg['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mg['raw_content'])} chars): "
      f"{res_mg['raw_content'][:500]!r}")
print()
if res_mg["validation_error"]:
    print(f"⚠ validation: {res_mg['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mg["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §12 `missile_airframe`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mai = call_ollama_direct("missile_airframe", doc)
print(f"-- elapsed: {res_mai['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mai['raw_thinking'])} chars): "
      f"{res_mai['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mai['raw_content'])} chars): "
      f"{res_mai['raw_content'][:500]!r}")
print()
if res_mai["validation_error"]:
    print(f"⚠ validation: {res_mai['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mai["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §13 `missile_speed_timing`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mst = call_ollama_direct("missile_speed_timing", doc)
print(f"-- elapsed: {res_mst['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mst['raw_thinking'])} chars): "
      f"{res_mst['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mst['raw_content'])} chars): "
      f"{res_mst['raw_content'][:500]!r}")
print()
if res_mst["validation_error"]:
    print(f"⚠ validation: {res_mst['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mst["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §14 `missile_propulsion`

Direct Ollama call. The cell prints timing, raw `thinking`, raw `content`, and the populated fields after Pydantic validation.

In [ ]:
res_mpr = call_ollama_direct("missile_propulsion", doc)
print(f"-- elapsed: {res_mpr['elapsed_seconds']:.1f}s --")
print(f"raw thinking ({len(res_mpr['raw_thinking'])} chars): "
      f"{res_mpr['raw_thinking'][:300]!r}")
print(f"raw content  ({len(res_mpr['raw_content'])} chars): "
      f"{res_mpr['raw_content'][:500]!r}")
print()
if res_mpr["validation_error"]:
    print(f"⚠ validation: {res_mpr['validation_error'][:200]}")
else:
    print("✓ validated against template")
parsed = res_mpr["parsed"] or {}
for system in (parsed.get("radar_systems") or []) + (parsed.get("missile_systems") or []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))


## §15 Rollup — direct-Ollama vs OllamaChatClient-via-docling-graph

Same canonicalize + merge logic as `extraction_walkthrough.ipynb` §16. Compare the entity / field counts here to that notebook to isolate the docling-graph service path (chunker, batcher, OllamaChatClient, identity filter, quality gate) as a variable. Both paths talk to the same Ollama backends — any divergence here vs the service is service-layer behavior, not Ollama behavior.


In [ ]:
import re
from collections import defaultdict


def _entities_from_result(res, list_key):
    """Extract entity dicts from one pass result.

    gpt-oss (and the docling-graph upstream library prompts in general)
    return graph node-link format: {"nodes": [{path, ids, properties}, ...]}.
    Production docling-graph runs a separate graph→template normalizer
    before Pydantic; in this direct-Ollama notebook we don't have that
    normalizer, so we do the equivalent manually here:

      1. If parsed dict has the flat list form (`radar_systems: [...]`),
         use it directly. (Some models emit this shape if the prompt's
         catalog block coaxes them.)
      2. Otherwise, walk `parsed.nodes[]` and pull entries whose
         `path == "<list_key>[]"`, merging their `ids` + `properties`
         into a flat dict.
      3. As a last fallback, return validated.model_dump()[list_key] —
         which will be `[]` for graph-format input but is correct for
         already-flat input.
    """
    parsed = res.get("parsed") or {}

    # 1. Already-flat shape
    flat = parsed.get(list_key)
    if isinstance(flat, list) and flat:
        return flat

    # 2. Graph-format walk — the production-equivalent normalization
    out = []
    list_path = f"{list_key}[]"
    for node in parsed.get("nodes", []) or []:
        if node.get("path") == list_path:
            entity = dict(node.get("properties") or {})
            entity.update(node.get("ids") or {})
            out.append(entity)
    if out:
        return out

    # 3. Last fallback — validated dump (will be [] for graph-format)
    validated = res.get("validated")
    if validated is not None:
        try:
            dumped = validated.model_dump()
        except Exception:
            dumped = None
        if isinstance(dumped, dict):
            return dumped.get(list_key) or []
    return []


def _identity_token_bag(entity):
    parts = [entity.get(f) for f in ("system_name", "nomenclature", "name")
             if isinstance(entity.get(f), str)]
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", " ".join(p for p in parts if p))
            if len(t) >= 2}


def _name_only_tokens(entity):
    name = entity.get("system_name") or ""
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", name) if len(t) >= 2}


def _likely_same_entity(a, b):
    a_name, b_name = a.get("system_name"), b.get("system_name")
    if not a_name or not b_name:
        return False
    if a_name == b_name:
        return True
    a_tokens = _name_only_tokens(a)
    b_tokens = _name_only_tokens(b)
    if not a_tokens or not b_tokens:
        return False
    return a_tokens.issubset(_identity_token_bag(b)) or \
           b_tokens.issubset(_identity_token_bag(a))


def _pick_canonical(names):
    cands = sorted({n for n in names if isinstance(n, str) and n}, key=lambda n: (len(n), n))
    return cands[0] if cands else ""


def canonicalize_systems(per_pass_lists):
    flat = [e for sub in per_pass_lists for e in sub]
    if len(flat) <= 1:
        return
    n = len(flat)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    for i in range(n):
        for j in range(i + 1, n):
            if _likely_same_entity(flat[i], flat[j]):
                union(i, j)
    components = defaultdict(list)
    for i in range(n):
        components[find(i)].append(i)
    rewrites = 0
    for member_idxs in components.values():
        if len(member_idxs) <= 1:
            continue
        canonical = _pick_canonical([flat[i].get("system_name") for i in member_idxs])
        if not canonical:
            continue
        for i in member_idxs:
            if flat[i].get("system_name") != canonical:
                flat[i]["system_name"] = canonical
                rewrites += 1
    if rewrites:
        print(f"canonicalize_systems: rewrote {rewrites} system_name(s)")


# Gather per-pass results.
RADAR_RESULTS = [
    ("identity",   res_rid),  ("power_rf",   res_rpr),
    ("antenna",    res_ran),  ("timing",     res_rt),
    ("modulation", res_rm),
]
MISSILE_RESULTS = [
    ("identity",     res_mid),  ("kinematics",   res_mki),
    ("guidance",     res_mg),   ("airframe",     res_mai),
    ("speed_timing", res_mst),  ("propulsion",   res_mpr),
]

radar_lists   = [_entities_from_result(r, "radar_systems")   for _, r in RADAR_RESULTS]
missile_lists = [_entities_from_result(r, "missile_systems") for _, r in MISSILE_RESULTS]

canonicalize_systems(radar_lists)
canonicalize_systems(missile_lists)

merged_radar   = defaultdict(dict)
merged_missile = defaultdict(dict)

for (label, _), entities in zip(RADAR_RESULTS, radar_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_radar[name][k] = v
        merged_radar[name].setdefault("_passes_seen_in", set()).add(label)

for (label, _), entities in zip(MISSILE_RESULTS, missile_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_missile[name][k] = v
        merged_missile[name].setdefault("_passes_seen_in", set()).add(label)


def _print_block(label, merged):
    print(f"\n=========== {label} ===========")
    for name, fields in merged.items():
        passes = sorted(fields.pop("_passes_seen_in", set()))
        print(f"\n--- {name} (seen in: {', '.join(passes)}) ---")
        for k in sorted(fields.keys()):
            print(f"  {k:<28} = {fields[k]}")


_print_block("RADAR_SYSTEMS", merged_radar)
_print_block("MISSILE_SYSTEMS", merged_missile)


all_results = RADAR_RESULTS + MISSILE_RESULTS
summary = {
    "model":     TARGET_MODEL,
    "think":     TARGET_THINK,
    "format":    FORMAT_MODE,
    "radar_systems_extracted":   list(merged_radar.keys()),
    "missile_systems_extracted": list(merged_missile.keys()),
    "total_populated_radar_fields":   sum(len(f) for f in merged_radar.values()),
    "total_populated_missile_fields": sum(len(f) for f in merged_missile.values()),
    "per_pass_elapsed_seconds": {
        f"{group_kind}_{label}": round(r["elapsed_seconds"], 1)
        for group_kind, results in [("radar", RADAR_RESULTS), ("missile", MISSILE_RESULTS)]
        for label, r in results
    },
    "validation_errors": {
        f"{group_kind}_{label}": r["validation_error"]
        for group_kind, results in [("radar", RADAR_RESULTS), ("missile", MISSILE_RESULTS)]
        for label, r in results
        if r["validation_error"]
    },
}
print()
print(json.dumps(summary, indent=2))
